<a href="https://colab.research.google.com/github/josephmedina132/Trabajo-final./blob/main/trabajo_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

import numpy as np

# --- 1. Definición de Parámetros ---
POP_SIZE = 100    # Tamaño de la población
N_BITS = 5        # Número de bits del cromosoma (x de 0 a 31)
GEN_MAX = 50      # Número máximo de generaciones
PROB_MUT = 0.01   # Probabilidad de mutación
PROB_CROSS = 0.7  # Probabilidad de cruce

# --- 2. Función Objetivo (Fitness) ---
# Se busca maximizar f(x) = x^2, donde x es el valor entero del cromosoma
def calculate_fitness(pop):
    # Decodificar el cromosoma binario a valor entero 'x'
    weights = 2**np.arange(N_BITS)[::-1]
    x = np.dot(pop, weights)
    
    # Calcular el valor de fitness
    return x**2, x

# --- 3. Operadores Genéticos ---

# a) Inicialización de la Población
def initialize_population():
    return np.random.randint(0, 2, size=(POP_SIZE, N_BITS))

# b) Selección por Ruleta
def select_parents(pop, fitness):
    # Ajustar fitness si hay valores negativos (no aplica aquí, pero es buena práctica)
    if np.min(fitness) < 0:
        norm_fitness = fitness - np.min(fitness)
    else:
        norm_fitness = fitness
        
    total_fitness = np.sum(norm_fitness)
    probabilities = norm_fitness / total_fitness # Cálculo de probabilidades (ruleta)
    
    # Seleccionar POP_SIZE de individuos con reemplazo
    indices = np.random.choice(np.arange(POP_SIZE), size=POP_SIZE, p=probabilities)
    return pop[indices]

# c) Cruce (Crossover) en un punto aleatorio
def crossover(p1, p2):
    if np.random.rand() < PROB_CROSS:
        cross_point = np.random.randint(1, N_BITS)
        o1 = np.concatenate((p1[:cross_point], p2[cross_point:]))
        o2 = np.concatenate((p2[:cross_point], p1[cross_point:]))
        return o1, o2
    return p1, p2

# d) Mutación (Bit-flip)
def mutate(chromosome):
    for i in range(N_BITS):
        if np.random.rand() < PROB_MUT:
            chromosome[i] = 1 - chromosome[i] # Voltear el bit
    return chromosome

# --- 4. Algoritmo Genético Principal ---
def genetic_algorithm():
    population = initialize_population()
    
    for gen in range(GEN_MAX):
        # 1. Evaluación
        fitness, x_values = calculate_fitness(population)
        
        # 2. Selección
        parents = select_parents(population, fitness)
        
        # 3. Reproducción (Cruce y Mutación)
        new_population = []
        for i in range(0, POP_SIZE, 2):
            p1, p2 = parents[i], parents[i+1]
            o1, o2 = crossover(p1, p2)
            o1 = mutate(o1)
            o2 = mutate(o2)
            new_population.extend([o1, o2])
            
        population = np.array(new_population)

    # Retorno del mejor individuo de la última población
    final_fitness, final_x = calculate_fitness(population)
    overall_best_index = np.argmax(final_fitness)
    return final_x[overall_best_index], final_fitness[overall_best_index]

# Ejecutar el algoritmo
# best_x, best_fitness = genetic_algorithm()

import tkinter as tk
from tkinter import ttk, messagebox
import calendar, datetime as dt

# -------- lógica simple --------
def estado_dia(d, ini, fin, trab, desc):
    if d < ini or d > fin: return "X"           # fuera de rango
    ciclo = trab + desc
    return "T" if ((d - ini) % ciclo) < trab else "D"

# -------- GUI --------
root = tk.Tk(); root.title("Planificador de turnos")

frm = ttk.Frame(root, padding=6); frm.grid()

# Entradas básicas
ttk.Label(frm, text="Mes").grid(row=0, column=0);  ent_mes = ttk.Entry(frm, width=4);  ent_mes.grid(row=0, column=1)
ttk.Label(frm, text="Año").grid(row=0, column=2);  ent_a  = ttk.Entry(frm, width=6);  ent_a.grid(row=0, column=3)
hoy = dt.date.today(); ent_mes.insert(0, hoy.month); ent_a.insert(0, hoy.year)

ttk.Label(frm, text="Modalidad").grid(row=1, column=0)
modo = tk.StringVar(value="4x4")
ttk.OptionMenu(frm, modo, "4x4", "4x4", "7x7", "14x14", "Personalizado").grid(row=1, column=1, columnspan=3, sticky="we")

ttk.Label(frm, text="Trabajo").grid(row=2, column=0); ent_t = ttk.Entry(frm, width=4); ent_t.insert(0,"4"); ent_t.grid(row=2, column=1)
ttk.Label(frm, text="Descanso").grid(row=2, column=2); ent_d = ttk.Entry(frm, width=4); ent_d.insert(0,"4"); ent_d.grid(row=2, column=3)

ttk.Label(frm, text="Inicio").grid(row=3, column=0);  ent_i = ttk.Entry(frm, width=4); ent_i.insert(0,"1"); ent_i.grid(row=3, column=1)
ttk.Label(frm, text="Término (0=fin)").grid(row=3, column=2); ent_f = ttk.Entry(frm, width=4); ent_f.insert(0,"0"); ent_f.grid(row=3, column=3)

# Calendario (6x7 etiquetas)
dias = ["Lun","Mar","Mié","Jue","Vie","Sáb","Dom"]
for c,n in enumerate(dias): ttk.Label(frm, text=n, width=6).grid(row=5, column=c)
celdas = [[tk.Label(frm, width=6, height=2, relief="ridge") for c in range(7)] for r in range(6)]
for r in range(6):
    for c in range(7): celdas[r][c].grid(row=6+r, column=c, padx=1, pady=1)

def aplicar_modo():
    if modo.get() != "Personalizado":
        t, d = map(int, modo.get().split("x")); ent_t.delete(0, tk.END); ent_t.insert(0, t)
        ent_d.delete(0, tk.END); ent_d.insert(0, d)

def generar():
    try:
        m, a = int(ent_mes.get()), int(ent_a.get())
        t, d = int(ent_t.get()), int(ent_d.get());
        _, ultimo = calendar.monthrange(a, m)
        i = int(ent_i.get()); f = int(ent_f.get()) or ultimo
        if not(1<=m<=12 and 1900<=a<=2100 and 1<=i<=ultimo and i<=f<=ultimo and t>0 and d>=0):
            raise ValueError
    except:
        return messagebox.showerror("Error", "Verifica tus datos.")
    aplicar_modo()
    mat = calendar.monthcalendar(a, m)  # semanas x 7
    for r in range(6):
        for c in range(7):
            dia = mat[r][c]
            lbl = celdas[r][c]
            if dia==0:
                lbl.config(text="", bg="white"); continue
            est = estado_dia(dia, i, f, int(ent_t.get()), int(ent_d.get()))
            bg = "lightgreen" if est=="T" else ("salmon" if est=="D" else "lightgray")
            lbl.config(text=str(dia), bg=bg)

def limpiar():
    for r in celdas:
        for lbl in r: lbl.config(text="", bg="white")

# Botones
ttk.Button(frm, text="Generar", command=generar).grid(row=4, column=0, columnspan=2, sticky="we", pady=4)
ttk.Button(frm, text="Limpiar",  command=limpiar ).grid(row=4, column=2, columnspan=2, sticky="we", pady=4)

root.mainloop()


# --- 1. Definición de Clases (POO) ---
class Zona:
    def __init__(self, nombre, habitantes):
        self.nombre = nombre
        self.habitantes = habitantes

class Pais:
    def __init__(self, nombre):
        self.nombre = nombre
        self.zonas = []

    def agregar(self, zona):
        self.zonas.append(zona)

    def graficar(self):
        # Extraer datos de los objetos
        nombres = [z.nombre for z in self.zonas]
        valores = [z.habitantes for z in self.zonas]

        # Crear gráfico (Matplotlib)
        import matplotlib.pyplot as plt
        plt.figure(figsize=(8, 5))
        plt.bar(nombres, valores, color=['#ff9999', '#66b3ff', '#99ff99'])

        # Etiquetas obligatorias
        plt.title(f"Población de {self.nombre} por Zona")
        plt.xlabel("Regiones")
        plt.ylabel("Habitantes")
        plt.show()

# --- 2. Ejecución ---
if __name__ == "__main__":
    chile = Pais("Chile")

    # Agregando las 3 zonas requeridas
    chile.agregar(Zona("Norte", 2500000))
    chile.agregar(Zona("Centro", 13000000))
    chile.agregar(Zona("Sur", 4000000))

    chile.graficar()